# Training YOLO11-seg on Non_AI pseudo-labels (Google Colab, T4 GPU)

The Non_AI pipeline produces one mask per object. Those masks are cut into 640 x 640 tiles by `make_dataset.py` and used as training labels, so the network learns to segment screws, bolts and nuts without any hand-drawn polygons. Five images are held out completely (see `make_dataset.py`).

Runtime: **GPU (T4)**.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics==8.3.40
# Colab ships NumPy 2.x, which removed np.trapz; ultralytics 8.3.40 still calls it
import numpy as np
np.trapz = np.trapezoid

## 1. Upload `dataset.zip` (made locally by `make_dataset.py`)

In [ ]:
from google.colab import files
import os, zipfile
if not os.path.exists('dataset.zip'):
    files.upload()
zipfile.ZipFile('dataset.zip').extractall('dataset')
open('dataset/data.yaml', 'w').write('path: /content/dataset
train: images/train
val: images/val
names:
  0: object
')
print(len(os.listdir('dataset/images/train')), 'train tiles,', len(os.listdir('dataset/images/val')), 'val tiles')

## 2. Train

* `yolo11s-seg` pretrained on COCO, single class `object`.
* Parts lie in every orientation, so rotation (±180°) and both flips are turned on.
* `scale=0.5`: the sparse photos show screws about 2x larger than the dense ones.
* `max_det=400`: a single tile of the nut piles can hold well over 100 objects.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s-seg.pt')
results = model.train(
    data='/content/dataset/data.yaml', imgsz=640, epochs=100, patience=30, batch=16,
    degrees=180, flipud=0.5, fliplr=0.5, scale=0.5, hsv_v=0.3, mosaic=1.0,
    close_mosaic=10, max_det=400, project='runs', name='counting', exist_ok=True, seed=0)

## 3. Validation metrics and training curves

In [ ]:
from IPython.display import Image, display
m = YOLO('runs/counting/weights/best.pt').val(data='/content/dataset/data.yaml', max_det=400)
print('mask mAP50:', round(m.seg.map50, 3), ' mask mAP50-95:', round(m.seg.map, 3))
display(Image('runs/counting/results.png', width=900))

## 4. Download the weights

`best.pt` goes into `AI/weights/` locally (git-ignored), where `count_objects_ai.py` runs SAHI sliced inference on the full images.

In [ ]:
import shutil
shutil.copy('runs/counting/weights/best.pt', 'best.pt')
shutil.make_archive('training_run', 'zip', 'runs/counting')
files.download('best.pt')
files.download('training_run.zip')